# 03 — Evaluation on ETHICS

We compare the baseline `Qwen2.5-1.5B-Instruct` against the RLHF-aligned policy from
notebook 02 on the ETHICS benchmark. Scoring is done via log-likelihood comparison
of two candidate continuations (the standard lm-evaluation-harness recipe), which is
much more stable than free-form generation parsing.

Each subset is sub-sampled to 100 examples from the **test** split. No ETHICS data
was used during training.

**Output:** `outputs/eval/results.json` and `outputs/eval/comparison.png`.

In [ ]:
# !pip install -q -r ../requirements.txt

In [ ]:
import os, sys, json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('Working dir:', Path.cwd())

In [ ]:
import torch
import numpy as np
import pandas as pd

from src.utils.config import load_config
from src.utils.seed import seed_everything
from src.data.ethics import load_all_ethics
from src.evaluation.ethics_eval import (
    evaluate_all, summarize, load_causal_lm_for_eval,
)

assert torch.cuda.is_available(), 'A GPU is required to run the evaluation in a reasonable time.'

cfg = load_config('configs/config.yaml')
seed_everything(cfg['seed'])

## 1. Build the evaluation set

100 examples per subset; the same examples are reused across all models.

In [ ]:
eval_cfg = cfg['evaluation']
examples = load_all_ethics(
    subsets=tuple(eval_cfg['ethics_subsets']),
    num_samples=eval_cfg['samples_per_subset'],
    seed=cfg['seed'],
)
for name, exs in examples.items():
    print(f'{name:>16}: {len(exs)} examples  (positive rate = {np.mean([e.label for e in exs]):.2f})')

## 2. Evaluate each model in turn

We load models one at a time and free the GPU between runs — Colab T4 cannot hold
two 1.5B models simultaneously.

In [ ]:
import gc

def evaluate_one(name, adapter_path):
    print(f'\n=== Evaluating: {name} (adapter={adapter_path}) ===')
    model, tok = load_causal_lm_for_eval(cfg['base_model'], adapter_path)
    res = evaluate_all(model, tok, examples)
    summary = summarize(res)
    print(f'{name} accuracies: {summary}')
    # Free GPU memory before loading the next model.
    del model, tok
    gc.collect(); torch.cuda.empty_cache()
    return {'summary': summary, 'per_subset': {k: {'accuracy': v.accuracy, 'n': v.n} for k, v in res.items()}}

all_results = {}
for name, spec in eval_cfg['models'].items():
    all_results[name] = evaluate_one(name, spec.get('adapter_path'))

## 3. Save raw results

In [ ]:
eval_dir = Path(cfg['paths']['eval_dir'])
eval_dir.mkdir(parents=True, exist_ok=True)
with open(eval_dir / 'results.json', 'w') as fh:
    json.dump(all_results, fh, indent=2)
print('Wrote', eval_dir / 'results.json')

## 4. Side-by-side table

In [ ]:
rows = []
for model_name, payload in all_results.items():
    for subset, stats in payload['per_subset'].items():
        rows.append({'model': model_name, 'subset': subset, 'accuracy': stats['accuracy']})
df = pd.DataFrame(rows).pivot(index='subset', columns='model', values='accuracy')
df.loc['average'] = df.mean()
df['delta'] = df.get('rlhf', 0) - df.get('baseline', 0)
df

In [ ]:
df.to_csv(eval_dir / 'comparison.csv')
print('Wrote', eval_dir / 'comparison.csv')

## 5. Bar chart for the report

In [ ]:
import matplotlib.pyplot as plt

plot_df = df.drop(index='average', errors='ignore').drop(columns='delta', errors='ignore')
ax = plot_df.plot(kind='bar', figsize=(9, 5), width=0.75)
ax.set_ylabel('accuracy')
ax.set_ylim(0, 1)
ax.set_title('ETHICS accuracy — baseline vs. RLHF-aligned (Qwen2.5-1.5B)')
ax.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='chance')
ax.legend(loc='lower right')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(eval_dir / 'comparison.png', dpi=160)
plt.show()